In [0]:
dbutils.widgets.text("process_date", "2026-08-31")
dbutils.widgets.text("run_id", "")

process_date = dbutils.widgets.get("process_date")
run_id = dbutils.widgets.get("run_id")

In [0]:
source_path = (
    "abfss://landing@stchilecompradev.dfs.core.windows.net/"
    "chilecompra/licitaciones/incremental/"
    f"process_date={process_date}/"
    f"run_id={run_id}/"
    "licitaciones.json"
)

In [0]:
df_raw = (
    spark.read
    .option("multiLine", "true")
    .json(source_path)
)


In [0]:
from pyspark.sql.functions import col, explode, lit, current_timestamp, to_timestamp

df_ls = (
    df_raw
    .select(
        explode(col("Listado")).alias("ls"),
        col("Cantidad").alias("_source_reported_count"),
        col("FechaCreacion").alias("_source_created_at"),
        col("Version").alias("_source_version")
    )
)

In [0]:
df_bronze = (
    df_ls
    .select(
        col("ls.CodigoExterno").alias("codigo_externo"),
        col("ls.CodigoEstado").cast("int").alias("codigo_estado"),
        to_timestamp(col("ls.FechaCierre")).alias("fecha_cierre"),
        col("ls.Nombre").alias("nombre"),

        col("_source_reported_count"),
        to_timestamp(col("_source_created_at")).alias("_source_created_at"),
        col("_source_version")
    )
    .withColumn("_process_date", lit(process_date).cast("date"))
    .withColumn("_pipeline_run_id", lit(run_id))
    .withColumn("_ingestion_timestamp", current_timestamp())
)

In [0]:
expected_count = df_raw.select("Cantidad").first()["Cantidad"]
actual_count = df_bronze.count()

null_codigo_count = (
    df_bronze
    .filter(col("codigo_externo").isNull())
    .count()
)

distinct_codigo_count = (
    df_bronze
    .select("codigo_externo")
    .distinct()
    .count()
)

null_estado_count = (
    df_bronze
    .filter(col("codigo_estado").isNull())
    .count()
)
if expected_count != actual_count:
    raise ValueError(
        f"DQ FAILED: API reported {expected_count} rows "
        f"but Bronze has {actual_count}"
    )

if null_codigo_count > 0:
    raise ValueError(
        f"DQ FAILED: {null_codigo_count} rows have codigo_externo NULL"
    )

duplicate_rows = actual_count - distinct_codigo_count

print("DQ checks passed")
print("duplicate codigo_externo rows:", duplicate_rows)
print("null codigo_estado:", null_estado_count)


In [0]:
target_table = "chilecompra.bronze.licitaciones_api"

if spark.catalog.tableExists(target_table):
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_process_date = DATE '{process_date}'"
        )
        .saveAsTable(target_table)
    )
else:
    (
        df_bronze.write
        .format("delta")
        .partitionBy("_process_date")
        .saveAsTable(target_table)
    )

In [0]:
%sql
SELECT
    _process_date,
    COUNT(*) AS rows,
    COUNT(DISTINCT codigo_externo) AS distinct_licitaciones
FROM chilecompra.bronze.licitaciones_api
GROUP BY _process_date
ORDER BY _process_date;